# S10 · Random Forest and XGBoost — many trees beat one

One tree is easy to read but wobbly. Here we combine many trees two different
ways and watch the accuracy jump. First a **Random Forest**: grow many trees and
let them vote. Then **XGBoost**: grow trees one after another, each fixing the
last one's mistakes. Then we read off which inputs mattered most.

**New here? Read this once.**

- New to Python? Run each cell top to bottom and read the note above it. You do
  not need to write any code to follow the whole story.
- New to the idea of combining models? The two pictures are simple: *many trees
  vote*, and *each tree fixes the last one*. Everything below is those two ideas.
- Already confident? Look for the cell marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

Run the next cell to load the libraries and datasets. On **Google Colab** this
also installs XGBoost, the one library Colab may not already have. On your **own
machine** you installed everything with `uv`, so it does nothing there.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colab may need xgboost installed; your own machine already has it.
if "google.colab" in sys.modules:
    !pip install -q xgboost
else:
    print("Not on Colab - assuming the libraries are already installed.")


def load(name):
    """Load one of this session's teaching datasets."""
    local = Path("../data") / f"{name}.csv"
    if local.exists():
        return pd.read_csv(local)
    return pd.read_csv(f"/content/{name}.csv")


print("Setup complete.")

## Step 1 — meet the noisier dataset

A single tree looked fine on two inputs. To see the ensembles earn their keep we
need a harder problem. `forest_noise_data` is a made-up loan-style dataset with
two inputs and a fair bit of noise: the two classes overlap, so no single tree
draws a clean flowchart.

In [ ]:
data = load("forest_noise_data")
data.head()

## Step 2 — split into a training part and a test part

Same honest habit as always: learn on one part, judge on a hidden part. Every
score we report from here on is on applicants the model never saw during training.

In [ ]:
from sklearn.model_selection import train_test_split

X = data[["feature_1", "feature_2"]]
y = data["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print("training applicants:", X_train.shape[0])
print("test applicants    :", X_test.shape[0])

## Step 3 — one decision tree (our baseline)

First we fit a single tree, exactly like the previous notebook. This is the score
the ensembles have to beat.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

single_tree = DecisionTreeClassifier(max_depth=8, random_state=42)
single_tree.fit(X_train, y_train)

single_tree_accuracy = single_tree.score(X_test, y_test)
print("single decision tree, test accuracy:", round(single_tree_accuracy, 3))

## Step 4 — a Random Forest (many trees vote)

Here is the first idea. **Bagging** means: build many trees, each on a different
random sample of the applicants, then let them vote on each new person. A **Random
Forest** adds one twist: each tree may only look at a random handful of the inputs
at each question. That keeps the trees different from one another, and averaging
many different opinions cancels out the mistakes any single tree makes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators = how many trees are in the forest.
forest = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
forest.fit(X_train, y_train)

forest_accuracy = forest.score(X_test, y_test)
print("random forest, test accuracy:", round(forest_accuracy, 3))

## Step 5 — more trees, steadier predictions?

A single tree is just a forest with one member. As we add trees, the vote should
steady and the test score should climb — quickly at first, then flattening out.

In [ ]:
tree_counts = [1, 10, 50, 100, 300]
forest_scores = []

for how_many_trees in tree_counts:
    f = RandomForestClassifier(n_estimators=how_many_trees,
                               max_depth=8, random_state=42)
    f.fit(X_train, y_train)
    forest_scores.append(f.score(X_test, y_test))

print("trees | test accuracy")
for how_many_trees, score in zip(tree_counts, forest_scores):
    print("  ", how_many_trees, "|", round(score, 3))

## Step 6 — XGBoost (each tree fixes the last one)

Here is the second idea. Instead of voting, **boosting** builds the trees **one
after another**. Each new tree is trained to fix the mistakes the earlier trees
are still making. The final prediction adds up all the trees together. XGBoost is
the most popular tool that does this, and on tables of business data it is often
the one to beat.

We switch to `xgb_credit_data`, a richer six-input credit dataset, so there is
more for boosting to learn and more for us to rank.

In [ ]:
credit = load("xgb_credit_data")
credit.head()

In [ ]:
Xc = credit.drop(columns=["default"])
yc = credit["default"]
feature_names = list(Xc.columns)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc)

print("training rows:", Xc_train.shape[0], " test rows:", Xc_test.shape[0])

In [ ]:
from xgboost import XGBClassifier

# Fit a single tree and a Random Forest on the same data, for comparison.
tree_c = DecisionTreeClassifier(max_depth=6, random_state=42)
tree_c.fit(Xc_train, yc_train)

forest_c = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
forest_c.fit(Xc_train, yc_train)

# n_estimators = how many trees we add, one after another.
# learning_rate = how big a correction each new tree is allowed to make.
booster = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss")
booster.fit(Xc_train, yc_train)

model_names = ["single tree", "random forest", "xgboost"]
model_scores = [tree_c.score(Xc_test, yc_test),
                forest_c.score(Xc_test, yc_test),
                booster.score(Xc_test, yc_test)]

print("model          | test accuracy")
for name, score in zip(model_names, model_scores):
    print(name.ljust(14), "|", round(score, 3))

## Step 7 — which inputs mattered most?

An ensemble can tell us how much each input contributed to its decisions, added up
across all its trees. This is called **feature importance**, and it answers the
first question any manager asks: *what is actually driving the decision?* Here are
the forest's importances, sorted, as a bar chart.

In [ ]:
importances = forest_c.feature_importances_
order = np.argsort(importances)

sorted_names = [feature_names[i] for i in order]
sorted_values = [importances[i] for i in order]

plt.figure(figsize=(8, 5))
plt.barh(sorted_names, sorted_values, color="#2E75B6")
plt.xlabel("relative importance")
plt.title("Random Forest feature importance")
plt.show()

print("input             | importance")
for i in order[::-1]:
    print(feature_names[i].ljust(17), "|", round(importances[i], 3))

## Step 8 — bagging vs boosting, one sentence each

A quick recap so the two ideas stay separate in your head.

In [ ]:
print("BAGGING (Random Forest):")
print("  - Build many trees in PARALLEL, each on a random sample of the data.")
print("  - They vote. Averaging cancels out their random mistakes.")
print("  - Steadies a single tree's wobble.")
print()
print("BOOSTING (XGBoost):")
print("  - Build trees ONE AFTER ANOTHER, each fixing the last one's errors.")
print("  - The final prediction is the SUM of all the trees.")
print("  - Usually the strongest on tables of business data.")

### Stretch (optional) — watch boosting improve as trees are added

Skip this if you are new to code. For the curious: boosting's prediction is a
**sum** of small trees, so adding more trees should keep nudging the score up (up
to a point). We refit XGBoost with a growing number of trees and plot the test
accuracy. This is the "additive model" idea made visible: many small corrections,
stacked up.

In [ ]:
boosting_counts = [1, 5, 10, 25, 50, 100, 200]
boosting_scores = []

for how_many_trees in boosting_counts:
    staged_booster = XGBClassifier(
        n_estimators=how_many_trees,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss")
    staged_booster.fit(Xc_train, yc_train)
    boosting_scores.append(staged_booster.score(Xc_test, yc_test))

plt.figure(figsize=(8, 5))
plt.plot(boosting_counts, boosting_scores, color="#27AE60", marker="o")
plt.xlabel("number of trees added (n_estimators)")
plt.ylabel("test accuracy")
plt.title("Boosting: each added tree is one more small correction")
plt.show()

for how_many_trees, score in zip(boosting_counts, boosting_scores):
    print("trees:", str(how_many_trees).rjust(3), " test accuracy:", round(score, 3))

## What you just did

You beat a single decision tree two ways. A **Random Forest** grows many
independent trees and lets them vote (bagging). **XGBoost** grows trees in
sequence, each correcting the last, and adds them up (boosting). Both are the
workhorses of real-world tabular machine learning. You also read off which inputs
the model leaned on, a first step toward understanding, and auditing, what a model
is really doing.

That audit is the whole point of the next notebook, where the decision affects
real people. Open `03_credit_default_and_bias_audit.ipynb`.